# Argumentation dialectique (ADF) : les conditions d'acceptation

**Navigation** : [README](./README.md) | [<< 03c](./Argumentation-03c-Ranking-Semantics-Python.ipynb) | [04 >>](./Argumentation-04-Dialogues-Protocolises-Python.ipynb)

| Champ | Valeur |
|---|---|
| **Module** | SymbolicAI / Argument_Analysis |
| **Rung** | 3d-formel (Dung et ses generalizations) |
| **Niveau** | Avance (semantiques formelles, trois valeurs) |
| **Duree estimee** | 40 minutes |
| **Kernel** | Python 3 (JPype + JVM JDK 17 + jars Tweety, module `adf-1.30.jar`) |

## Objectifs

Le carnet 03 a montre le cadre de **Dung** : un graphe d'arguments et une
relation d'attaque binaire. Ce carnet montre ce que les **Abstract Dialectical
Frameworks** (ADF) ajoutent a ce cadre, et il le fait **en executant l'organe
natif** de la bibliotheque `argumentation_lib` : aucune semantique n'est
reimplementee ici.

A la fin de ce carnet, vous saurez :

- traduire une **condition d'acceptation** en ADF, et construire un cadre ;
- lire une **interpretation a trois valeurs** (`t` / `f` / `u`) ;
- nommer les deux ecarts qui separent l'ADF de Dung, et les **voir dans la
  sortie** plutot que de les postuler.

## Prerequis

Le carnet [`Argumentation-00-Setup-Tweety-Python`](./Argumentation-00-Setup-Tweety-Python.ipynb)
a amorce la JVM ; ce carnet reutilise la meme passerelle et le meme
classpath (jar `adf-1.30.jar` inclus).

## 1. Ce qu'un ADF ajoute a Dung

En Dung, un argument est **accepte** ou **refuse**, et il est refuse parce
qu'un autre argument l'attaque. La relation d'attaque est **binaire** : soit
`x` attaque `y`, soit non.

Les **ADF** remplacent cette relation par une **condition d'acceptation** :
pour chaque argument, une formule booleenne sur les arguments dont il depend
(ses « parents »).

| | Dung | ADF |
|---|---|---|
| relation | attaque binaire | condition d'acceptation booleenne |
| etat d'un argument | accepte / refuse | accepte / refuse / **indecis** |
| « refuse, mais par qui ? » | toujours un attaquant | condition de contradiction : aucun |
| forme du resultat | extension (deux valeurs) | interpretation (trois valeurs) |

Les deux dernieres lignes sont les differences qui comptent, et ce carnet les
**mesure** : elles se lisent dans les sorties de l'organe, elles ne se
postulent pas.

In [1]:
import sys

import jpype

from argumentation_lib import initialize_jvm

# La JVM est amorcee par la shim de la bibliotheque (classpath Tweety complet,
# JDK 17 portable). Aucun garde fatal dans cette cellule (regle C.1) : si
# l'amorcage echoue, les cellules suivantes echoueront bruyamment, ce qui est
# preferable a un mode degrade qui simulerait un solveur formel.
JVM_OK = initialize_jvm(verbose=False)

print(f"Python : {sys.version.split()[0]}")
print(f"JPype  : {jpype.__version__}")
print(f"JVM operationnelle : {jpype.isJVMStarted()} (initialize_jvm -> {JVM_OK})")

# Preuve que le module ADF du jar vende est expose a la JVM.
ADF_CLASS = jpype.JClass("org.tweetyproject.arg.adf.syntax.adf.GraphAbstractDialecticalFramework")
print(f"Module ADF du jar expose : {ADF_CLASS is not None}")

Python : 3.12.13
JPype  : 1.6.0
JVM operationnelle : True (initialize_jvm -> True)
Module ADF du jar expose : True


## 2. L'organe que l'on interroge

Tout le calcul passe par l'organe natif `argumentation_lib._adf_handler.ADFHandler`,
atteint par l'acceseur public `get_adf_handler_symbol()`. Il interroge le
**vrai solveur** embarque dans `adf-1.30.jar` (les reasoners ADF sont SAT-based ;
le jar livre les solveurs natifs `minisat.dll` / `picosat.dll` / `lingeling.dll`).

L'API est celle de la bibliotheque :

```
analyze_adf(statements, acceptance_conditions, semantics) -> dict
```

- `statements` : la liste des noms d'arguments du cadre ;
- `acceptance_conditions` : un dictionnaire `argument -> condition` ;
- `semantics` : `grounded`, `preferred`, `model`, `complete`, `admissible`,
  `naive` ou `conflict_free`.

Le vocabulaire des conditions est le suivant :

| Condition | Signification |
|---|---|
| `tautology` | l'argument est toujours acceptable |
| `contradiction` | l'argument n'est jamais acceptable |
| `negation:<stmt>` | l'argument est acceptable si et seulement si `<stmt>` ne l'est pas |
| `or:<c1>,<c2>` | disjonction de deux conditions |
| `and:<c1>,<c2>` | conjonction de deux conditions |

Un argument absent du dictionnaire recoit la condition `tautology`.

In [2]:
from argumentation_lib import get_adf_handler_symbol

ADFHandler = get_adf_handler_symbol()
adf = ADFHandler()

print("Semantiques exposees par l'organe (jar ADF) :")
for nom, classe in ADFHandler.REASONERS.items():
    print(f"  - {nom:<14} {classe}")

print()
print("Vocabulaire des conditions d'acceptation utilise dans ce carnet :")
for exemple in ("tautology", "contradiction", "negation:<stmt>",
                "or:<c1>,<c2>", "and:<c1>,<c2>"):
    print(f"  - {exemple}")

print()
print("Type de l'organe instancie :", type(adf).__name__)

Semantiques exposees par l'organe (jar ADF) :
  - grounded       org.tweetyproject.arg.adf.reasoner.GroundReasoner
  - complete       org.tweetyproject.arg.adf.reasoner.CompleteReasoner
  - preferred      org.tweetyproject.arg.adf.reasoner.PreferredReasoner
  - admissible     org.tweetyproject.arg.adf.reasoner.AdmissibleReasoner
  - model          org.tweetyproject.arg.adf.reasoner.ModelReasoner
  - naive          org.tweetyproject.arg.adf.reasoner.NaiveReasoner
  - conflict_free  org.tweetyproject.arg.adf.reasoner.ConflictFreeReasoner

Vocabulaire des conditions d'acceptation utilise dans ce carnet :
  - tautology
  - contradiction
  - negation:<stmt>
  - or:<c1>,<c2>
  - and:<c1>,<c2>

Type de l'organe instancie : ADFHandler


## 3. Premier cadre : deux arguments qui se nient mutuellement

Prenons deux arguments `a` et `b` dont chacun est accepte **si et seulement si
l'autre ne l'est pas** :

- `a` : condition `negation:b`
- `b` : condition `negation:a`

C'est le cadre minimal qui distingue les deux formalismes. On l'interroge avec
l'organe natif, sous trois semantiques.

In [3]:
CADRE_AB = {"a": "negation:b", "b": "negation:a"}

for semantique in ("grounded", "preferred", "model"):
    resultat = adf.analyze_adf(["a", "b"], CADRE_AB, semantique)
    print(f"--- ADF / {semantique} ---")
    for interpretation in resultat["interpretations_detailed"]:
        print(f"  {interpretation['text']:<20} "
              f"accepte={interpretation['accepted']} "
              f"rejete={interpretation['rejected']} "
              f"indecis={interpretation['undecided']}")

--- ADF / grounded ---
  { u(b) u(a)}         accepte=[] rejete=[] indecis=['a', 'b']
--- ADF / preferred ---
  {t(b) f(a)}          accepte=['b'] rejete=['a'] indecis=[]
  {t(a) f(b)}          accepte=['a'] rejete=['b'] indecis=[]
--- ADF / model ---
  {t(b) f(a)}          accepte=['b'] rejete=['a'] indecis=[]
  {t(a) f(b)}          accepte=['a'] rejete=['b'] indecis=[]


### Lecture du verdict

Trois lectures a faire.

1. **`grounded`** : les deux arguments sont **indecis** (`u`). La semantique
   grounded est le point fixe minimal : au premier tour, `a` serait accepte si
   `b` ne l'est pas — mais `b` n'est pas encore decide, donc rien ne se
   decroche. Le cadre ne permet pas de decider, et l'ADF **le dit** au lieu de
   trancher.
2. **`preferred`** : deux interpretations, chacune accepte exactement un des
   deux arguments et rejette l'autre. L'ADF retrouve les deux extensions
   preferees de Dung.
3. **`model`** : les memes deux interpretations — ici, les deux modeles sont
   **deux-valeurs** (aucun argument indécis).

Pour verifier que l'ecart porte bien sur la **semantique grounded** et pas sur
le classement, on interroge le meme cadre par l'**organe AF** (Dung) de la
meme bibliotheque.

In [4]:
# L'organe AF de la meme bibliotheque lit le meme cadre en Dung :
# `a` attaque `b` et `b` attaque `a` (la negation de l'acceptation).
from argumentation_lib import get_af_handler_symbol, get_tweety_initializer_symbol

initialiseur = get_tweety_initializer_symbol()()
initialiseur.ensure_jvm_and_components_are_ready()
initialiseur.initialize_af_components()
af = get_af_handler_symbol()(initialiseur)

ATTAQUES_AB = [["a", "b"], ["b", "a"]]

for semantique in ("grounded", "preferred"):
    resultat = af.analyze_dung_framework(["a", "b"], ATTAQUES_AB, semantique)
    print(f"--- Dung (organe AF) / {semantique} : {resultat['extensions'][semantique]}")

print()
print("Comparaison sur la semantique grounded :")
print("  Dung : extension vide (les deux arguments sont REFUSES)")
print("  ADF  : { u(a) u(b) } (les deux arguments sont INDECIS)")

--- Dung (organe AF) / grounded : [[]]
--- Dung (organe AF) / preferred : [['a'], ['b']]

Comparaison sur la semantique grounded :
  Dung : extension vide (les deux arguments sont REFUSES)
  ADF  : { u(a) u(b) } (les deux arguments sont INDECIS)


## 4. L'argument fantome : refuse sans attaquant

En Dung, un argument que personne n'attaque est **toujours** accepte : il
figure dans l'extension grounded. On ne peut pas dire « cet argument est
refuse » sans introduire un attaquant.

Un ADF le peut : une condition de **contradiction** signifie que l'argument
n'est jamais acceptable, meme si aucun autre argument ne le vise.

C'est le second ecart entre les deux formalismes, et ce n'est pas une subtilite
de lecture : c'est une **capacite d'expression**.

In [5]:
CADRE_FANTOME = {"x": "contradiction", "y": "tautology"}

for semantique in ("grounded", "model"):
    resultat = adf.analyze_adf(["x", "y"], CADRE_FANTOME, semantique)
    for interpretation in resultat["interpretations_detailed"]:
        print(f"--- ADF / {semantique} : {interpretation['text']:<16} "
              f"accepte={interpretation['accepted']} "
              f"rejete={interpretation['rejected']} "
              f"indecis={interpretation['undecided']}")

print()
print("--- Dung : le meme `x`, mais sans condition de contradiction ---")
resultat = af.analyze_dung_framework(["x", "y"], [], "grounded")
print(f"  grounded : {resultat['extensions']['grounded']}  "
      "(x n'est attaque par personne : il est accepte)")

print()
print("En Dung, un argument sans attaquant ne peut pas etre refuse.")

--- ADF / grounded : {t(y) f(x)}      accepte=['y'] rejete=['x'] indecis=[]
--- ADF / model : {t(y) f(x)}      accepte=['y'] rejete=['x'] indecis=[]

--- Dung : le meme `x`, mais sans condition de contradiction ---
  grounded : [['x', 'y']]  (x n'est attaque par personne : il est accepte)

En Dung, un argument sans attaquant ne peut pas etre refuse.


## 5. L'exemple canonique : une disjonction de conditions

On ajoute un troisieme argument `c`, accepte si `a` **ou** `b` n'est pas
accepté — condition `or:negation:a,negation:b`.

C'est l'exemple canonique des ADF : c'est le fichier `adf_example.txt` que le
jar lui-meme embarque. Il est instructif parce que `c` est accepte dans les
**deux** modeles : la disjonction rend `c` robuste au choix entre `a` et `b`,
ce qu'un simple classement ne montre pas.

In [6]:
CADRE_CANONIQUE = {
    "a": "negation:b",
    "b": "negation:a",
    "c": "or:negation:a,negation:b",
}

for semantique in ("grounded", "preferred", "model"):
    resultat = adf.analyze_adf(["a", "b", "c"], CADRE_CANONIQUE, semantique)
    print(f"--- ADF / {semantique} ---")
    for interpretation in resultat["interpretations_detailed"]:
        print(f"  {interpretation['text']:<22} "
              f"accepte={interpretation['accepted']} "
              f"rejete={interpretation['rejected']} "
              f"indecis={interpretation['undecided']}")

--- ADF / grounded ---
  { u(b) u(a) u(c)}      accepte=[] rejete=[] indecis=['a', 'b', 'c']
--- ADF / preferred ---
  {t(a) f(c) u(b)}       accepte=['a'] rejete=['c'] indecis=['b']
  {t(a) t(c) f(b)}       accepte=['a', 'c'] rejete=['b'] indecis=[]
  {t(b) t(c) f(a)}       accepte=['b', 'c'] rejete=['a'] indecis=[]
  {t(b) f(c) u(a)}       accepte=['b'] rejete=['c'] indecis=['a']
--- ADF / model ---
  {t(a) t(c) f(b)}       accepte=['a', 'c'] rejete=['b'] indecis=[]
  {t(b) t(c) f(a)}       accepte=['b', 'c'] rejete=['a'] indecis=[]


## 6. Lire les trois valeurs

Une interpretation ADF attribue a chaque argument une valeur :

- `t` : **accepte**
- `f` : **refuse**
- `u` : **indecis** (ni accepte, ni refuse)

`u` n'est pas une absence d'information : c'est une **absence de decision** de
la semantique. Le `grounded` est le point fixe minimal — les arguments qui ne
peuvent pas etre decides y restent `u`, la ou une semantique plus forte
(ici `model`) doit trancher et les passe en `t` ou en `f`.

C'est la difference de fond avec Dung : la semantique `grounded` de Dung est
une **extension** (deux valeurs), celle de l'ADF est une **interpretation**
(trois valeurs).

In [7]:
def tableau(interpretations_detailed):
    """Rend les interpretations sous forme de tableau markdown lisible."""
    lignes = ["| interpretation | acceptes | rejetes | indecis |", "|---|---|---|---|"]
    for interpretation in interpretations_detailed:
        accepte = interpretation["accepted"] or "—"
        rejete = interpretation["rejected"] or "—"
        indecis = interpretation["undecided"] or "—"
        lignes.append(f"| `{interpretation['text']}` | {accepte} | {rejete} | {indecis} |")
    return "\n".join(lignes)


CADRES = (
    ("a <-> b", ["a", "b"], CADRE_AB),
    ("canonique (a <-> b, c)", ["a", "b", "c"], CADRE_CANONIQUE),
    ("fantome (x, y)", ["x", "y"], CADRE_FANTOME),
)

for nom, arguments, conditions in CADRES:
    resultat = adf.analyze_adf(arguments, conditions, "grounded")
    print(f"### {nom} — grounded")
    print(tableau(resultat["interpretations_detailed"]))
    print()

### a <-> b — grounded
| interpretation | acceptes | rejetes | indecis |
|---|---|---|---|
| `{ u(b) u(a)}` | — | — | ['a', 'b'] |

### canonique (a <-> b, c) — grounded
| interpretation | acceptes | rejetes | indecis |
|---|---|---|---|
| `{ u(c) u(a) u(b)}` | — | — | ['a', 'b', 'c'] |

### fantome (x, y) — grounded
| interpretation | acceptes | rejetes | indecis |
|---|---|---|---|
| `{t(y) f(x)}` | ['y'] | ['x'] | — |



## 7. Exercices

Trois exercices, un par acquisition :

- **Exercice 1** — ecrire une condition d'acceptation **composee** : une
  conjonction de deux negations.
- **Exercice 2** — construire un ADF dont le `grounded` est **non vide**, et le
  comparer au meme cadre lu en Dung.
- **Exercice 3** — filtrer, parmi les interpretations d'une semantique, celles
  qui sont **deux-valeurs** (aucun argument indécis).

In [8]:
# Exercice 1 : ecrire la condition d'acceptation de `d`.
# TODO etudiant : `d` doit etre accepte si et seulement si `a` ET `b` sont
# tous deux REFUSES. Ecrire la chaine de la condition dans CONDITION_D.
# Indice : la negation de l'acceptation d'un argument s'ecrit "negation:<nom>" ;
#          deux conditions se combinent avec "and:".
# Etape 1 : completer CONDITION_D avec le vocabulaire de l'organe.
# Etape 2 : passer le cadre {"a": ..., "b": ..., "d": CONDITION_D} a analyze_adf
#           et verifier que `d` est accepte quand `a` et `b` sont refuses.
CONDITION_D = None  # TODO etudiant

print("Condition proposée pour d :", CONDITION_D)

Condition proposée pour d : None


In [9]:
# Exercice 2 : un ADF dont le `grounded` n'est pas vide.
# TODO etudiant : proposer un cadre de trois arguments (liste des noms +
# dictionnaire des conditions) dont la semantique grounded accepte au moins un
# argument, puis le comparer au meme cadre lu en Dung par l'organe AF.
# Indice : un argument dont la condition est "tautology" est toujours accepte ;
#          l'ecart entre Dung et ADF se lit sur les arguments "indecis".
# Etape 1 : completer CADRE_EXERCICE_2 (liste ET conditions).
# Etape 2 : passer CADRE_EXERCICE_2 a analyze_adf, puis a
#           af.analyze_dung_framework avec les attaques correspondantes.
CADRE_EXERCICE_2 = None  # TODO etudiant

print("Cadre proposé :", CADRE_EXERCICE_2)

Cadre proposé : None


In [10]:
# Exercice 3 : distinguer les interpretations deux-valeurs des interpretations partielles.
# TODO etudiant : completer la fonction pour qu'elle retourne la liste des
# interpretations dont AUCUN argument n'est "indecis" (donc les
# interpretations deux-valeurs, les seules que la semantique `model` retient).
# Indice : chaque element de la liste fournie est un dictionnaire avec les
#          cles "accepted", "rejected" et "undecided".
# Etape 1 : completer le corps de la fonction.
# Etape 2 : l'appliquer au cadre canonique sous la semantique "preferred".
def deux_valeurs(interpretations_detailed):
    resultat = None  # TODO etudiant
    return resultat


print("Interpretations deux-valeurs du cadre canonique :")
print(deux_valeurs(
    adf.analyze_adf(["a", "b", "c"], CADRE_CANONIQUE, "preferred")["interpretations_detailed"]
))

Interpretations deux-valeurs du cadre canonique :
None


## Conclusion

Ce carnet a montre trois choses.

1. Les **ADF** generalisent Dung : la relation d'attaque binaire devient une
   condition d'acceptation booleenne, et le resultat d'une semantique devient
   une **interpretation a trois valeurs**.
2. Les deux ecarts se **voient dans la sortie** : le `grounded` ADF laisse des
   arguments **indecis** la ou Dung les refuse tous, et un argument portant une
   condition de contradiction est **refuse sans attaquant**.
3. Tout vient de l'**organe natif** `argumentation_lib._adf_handler` : le
   rebrancher sur l'API reellement vended du jar `adf-1.30.jar` (les liens du
   graphe, les conditions devenues des enumerations, le solveur SAT des
   reasoners) est ce qui rend la voie accessible depuis un carnet. Les
   reponses affichees ici ne sont pas reimplementees : elles viennent du
   solveur.

Ce que l'organe ne fait pas encore : la lecture d'un fichier `.kpp` (format
KPPADF) n'est pas cablee contre ce jar, et `parse_adf_file` le dit
explicitement. La voie supportee est `analyze_adf`, qui construit le cadre en
Python.

Pour aller plus loin : le carnet 03 (Dung) pour le cadre d'attaque, le carnet
03b (AF value-based) pour le classement, et la matrice de richeresse formelle
du carnet 05c pour situer ce rung parmi les autres.